## Description

In this programming assignment, you are required to implement the k-means algorithm and apply it to a real-life data set. 

**Input**

The provided input file ("places.txt") consists of the locations of 300 places in the US. Each location is a two-dimensional point that represents the longitude and latitude of the place. For example, "-112.1,33.5" means the longitude of the place is -112.1, and the latitude is 33.5.

**Output**

You are required to implement the k-means algorithm and use it to cluster the 300 locations into three clusters, such that the locations in the same cluster are geographically close to each other.

After reading in the 300 locations in "places.txt" and applying the k-means algorithm (with k = 3), you are required to generate an output file named "clusters.txt".  The output file should contain exactly 300 lines, where each line represents the cluster label of each location.  Every line should be in the format: location_id cluster_label.

An example snippet of the output "clusters.txt" file is provided below:

0 1

1 0

2 1

3 2

4 0

In the above, the five lines denote the cluster ids of the first five locations in the input file, which means:

The first location belongs to cluster "1"

The second location belongs to cluster "0"

The third location belongs to cluster "1"

The fourth location belongs to cluster "2"

The fifth location belongs to cluster "0"

**Important Tips**

When implementing the K-means algorithm, you could use any programming language you like. We only need the generated cluster label file.

Make sure that you format each line correctly in the output file.  For instance, use space instead of comma to separate the data point id and the cluster label.

In the input file "places.txt", the id of the location starts from 0.  That is, the first line in "places.txt" has id 0, the second line has id 1, ..., and the last place has id 299.

When generating the output file, please note that the order of the cluster labels does not matter.  For example, if there are three clusters, you can use either [0, 1, 2] or [2, 1, 0] as labels for them --- it is correct as long as you use three distinct integer ids. Thus, the following two cases will be considered equivalent by the grader:

**Case 1:**

0 0

1 1

2 2

**Case 2:**

0 2

1 1

2 0

## Solution

The following cells:
1. Load the 300 points from places.txt.
2. Implement k-means from scratch with multiple random restarts.
3. Save the final labels into clusters.txt in required format: `location_id cluster_label`.

In [1]:
from pathlib import Path
import numpy as np

# Locate the input file robustly (works whether cwd is notebook folder or workspace root).
input_candidates = [
    Path("places.txt"),
    Path("coursera labs/Cluster Analysis in Data Mining W2_L3/places.txt"),
]

input_path = next((p for p in input_candidates if p.exists()), None)
if input_path is None:
    raise FileNotFoundError("Could not find places.txt")

points = np.loadtxt(input_path, delimiter=",", dtype=float)
if points.shape != (300, 2):
    print(f"Warning: expected shape (300, 2), got {points.shape}")


def kmeans(points: np.ndarray, k: int = 3, max_iter: int = 300, tol: float = 1e-6, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    n = points.shape[0]

    # Initialize centroids using random unique points.
    centroid_idx = rng.choice(n, size=k, replace=False)
    centroids = points[centroid_idx].copy()

    labels = np.zeros(n, dtype=int)

    for _ in range(max_iter):
        # Assign each point to the nearest centroid.
        dists = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)

        # Recompute centroids.
        new_centroids = centroids.copy()
        for j in range(k):
            cluster_points = points[new_labels == j]
            if len(cluster_points) > 0:
                new_centroids[j] = cluster_points.mean(axis=0)
            else:
                # Handle empty clusters by re-seeding to a random point.
                new_centroids[j] = points[rng.integers(0, n)]

        centroid_shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        labels = new_labels

        if centroid_shift <= tol:
            break

    # Inertia = sum of squared distances to assigned centroid.
    inertia = np.sum((points - centroids[labels]) ** 2)
    return labels, centroids, inertia


# Multiple restarts to avoid poor local minima.
best_labels = None
best_centroids = None
best_inertia = np.inf

for seed in range(30):
    labels, centroids, inertia = kmeans(points, k=3, random_state=seed)
    if inertia < best_inertia:
        best_inertia = inertia
        best_labels = labels
        best_centroids = centroids

# Save output in required format: location_id cluster_label
output_path = input_path.with_name("clusters.txt")
with output_path.open("w", encoding="utf-8") as f:
    for i, label in enumerate(best_labels):
        f.write(f"{i} {int(label)}\n")

print(f"Input file: {input_path}")
print(f"Output file created: {output_path}")
print(f"Best inertia: {best_inertia:.4f}")
print("Cluster sizes:", {c: int((best_labels == c).sum()) for c in range(3)})

Input file: places.txt
Output file created: clusters.txt
Best inertia: 0.2576
Cluster sizes: {0: 100, 1: 100, 2: 100}


In [2]:
# Quick format check: file must contain exactly 300 lines with "id label".
lines = output_path.read_text(encoding="utf-8").strip().splitlines()
print("Number of lines:", len(lines))
print("First 10 lines:")
for line in lines[:10]:
    print(line)

Number of lines: 300
First 10 lines:
0 0
1 0
2 0
3 0
4 2
5 2
6 1
7 0
8 0
9 1
